In [1]:
from dotenv import load_dotenv, find_dotenv
result = load_dotenv(find_dotenv())

In [2]:
from llama_index.llms.azure_openai import AzureOpenAI
import os
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

llm = AzureOpenAI(
    engine="gpt-4o-mini",
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_ENDPOINT"],
    api_version="2024-10-21",
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-ada-002",
    deployment_name="text-embedding-ada-002",
    api_key=os.environ["OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_ENDPOINT"],
    api_version="2024-10-21",
)


In [3]:
import chromadb

# Initialize the Chroma client
db = chromadb.PersistentClient(path="./chroma_db")

# Create or get an existing collection
chroma_collection = db.get_or_create_collection("llamaindex")


In [4]:
# pip install llama-index-vector-stores-chroma
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


In [5]:
from llama_index.core.settings import Settings

# Set global settings
Settings.llm = llm
Settings.embed_model = embed_model


In [6]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("/Users/gianmariaricci/Downloads/llchaindata").load_data()
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context)


In [9]:

# Query from saved 
db2 = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db2.get_or_create_collection("llamaindex")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_vector_store(
    vector_store,
    embed_model=embed_model,
)

# Query Data from the persisted index
query_engine = index.as_query_engine()
response = query_engine.query("What did the author do growing up?")
display(Markdown(f"<b>{response}</b>"))

ValueError: Expected where to have exactly one operator, got {} in query.

In [7]:
query_engine = index.as_query_engine()


In [8]:
response = query_engine.query("who is leonardo da vinci?")
print(response)

ValueError: Expected where to have exactly one operator, got {} in query.

In [ ]:
response = query_engine.query("Come posso usare sqlmap in C#?")
print(response)

In [ ]:
from pprint import pprint
pprint(index)

